# Run 8: Jev and SAM 3 with both parts' geometry, the fitting slid over a spanner

The chrome pipe fitting of run 7 stands on the table beside a standing box spanner. The arm takes the fitting
and slides it down over the spanner: the fitting's bore is 25.5 mm and the spanner's hex 21.3 mm across its
corners, about 2 mm a side. Both parts are chrome, so both are placed by their geometry: each SAM 3 mask of a
metal part is fitted with both STLs (`piper_llm/assets/objects/pipe-fitting` and `box-spanner`), and the
better fit names it. Jev picks each step; `SlideOver` in `piper_llm/task.py` carries it out.

# ⚠️ Before you run this

- E-stop within reach.
- Nobody inside the arm's reach.
- Table clear of anything breakable.
- `speed_percent` at 10 for the first run.
- The arm reaches 58 cm out in this run (the default workspace box stops at 50 cm): keep that space clear.

The model can pick a wrong skill and perception can be wrong. The checks in
`piper_llm/safety.py` reduce the risk; they do not remove it.

In [ ]:
import math, numpy as np
from dotenv import load_dotenv           # pip install python-dotenv
load_dotenv()                            # reads .env, which is not in git

from piper_llm.arm import Arm
from piper_llm.camera import RealSense, Frame
from piper_llm.kinematics import Kinematics
from piper_llm.safety import Governor
from piper_llm.skills import Runner, SLIDE_OVER_SKILLS
from piper_llm.task import SlideOver
from piper_llm.config import ArmConfig, SafetyLimits, SceneConfig
from piper_llm.deciders import Jev
from piper_llm.perception import Detector
from piper_llm.record import Recorder
from piper_llm.geometry import ObjectGeometry

In [ ]:
# Dry check: no torque. Read the arm, camera and models first.
cam = RealSense()
rgb, depth = cam.frames()
print("camera ok", rgb.shape, "depth range %.2f-%.2f m" % (depth[depth>0].min(), depth.max()))

frame = Frame(cam)                        # needs calibration.json
kin = Kinematics()                        # needs the Menagerie MJCF
print("kinematics ok")

In [ ]:
# Read-only arm check. The arm does not move in this cell.
arm = Arm(ArmConfig(speed_percent=10, gripper_effort_nm=2.0))   # smooth chrome: the gripper's full squeeze
print("joints", np.round(arm.joints(), 3))
print("gripper", round(arm.gripper(), 2))
print("fault:", arm.status_error())

## Check perception before moving

SAM 3 finds both parts as "metal object". Each mask should be named by the geometry that fits it better, both
standing, with outline matches above 0.85. The fitting stands on the table 50 to 60 cm from the arm's base,
at least 8 cm from the spanner, which stands 43 to 46 cm out so the camera sees the fitting over it.

In [ ]:
detector = Detector(backend="sam3")
fitting = ObjectGeometry.load("../piper_llm/assets/objects/pipe-fitting")
spanner = ObjectGeometry.load("../piper_llm/assets/objects/box-spanner", min_prob=0.0)   # keeps its standing poses
rgb, depth = cam.frames()
for d in detector.detect(rgb, ["metal object"]):
    if d.mask is None:
        continue
    fits = {"fitting": fitting.locate(d.mask, frame), "spanner": spanner.locate(d.mask, frame)}
    name = max(fits, key=lambda k: fits[k][1])
    pose, fit, _ = fits[name]
    print("%-8s outline match %.2f, standing %s, axis %s, %.0f cm from the base"
          % (name, fit, abs(pose[2, 2]) > 0.9, np.round(pose[:2, 3], 3), 100 * np.linalg.norm(pose[:2, 3])))
print("detector median %.0f ms" % detector.median_ms())

## Run

Each step: the task reads the camera and builds the state, Jev picks a skill, the governor checks it, and the
task carries it out. The gripper leans 45 degrees about the line between its fingers the whole run: the wrist
cannot point it straight down 20 cm up, and with the fingers level the fitting held upright stays upright. That
lean needs the fitting 50 to 60 cm out, hence the wider workspace box. Over the spanner, `align_over_spanner`
measures the held fitting against the spanner's axis with the camera and corrects until the two agree within
0.8 mm; `slide_down` then goes down at 5 mm/s while the depth camera watches the gap above the spanner's top.
A move that stops, a missed grasp, or a fitting that will not go onto the spanner ends the run, and the
fitting goes back where it was picked.

In [ ]:
rec = Recorder(cam, arm, "run8")       # colour, depth, joints and decisions to out/recordings
arm.home()
gov = Governor(limits=SafetyLimits(workspace_high=(0.58, 0.35, 0.40)))   # the fitting stands up to 58 cm out
runner = Runner(arm=arm, kin=kin, gov=gov, scene=SceneConfig())
task = SlideOver(detector, frame, runner, cam, fitting, spanner, rec=rec, lean=math.radians(45))
jev = Jev()

INSTRUCTIONS = ("Pick the next skill for a PiPER arm that must slide the pipe fitting down over the standing box "
                "spanner. Each skill says when it applies: pick the one whose condition matches the state. Order: "
                "approach, descend, close, lift, move over the spanner, align, slide down, let go, retreat, done.")

gov.reset()
for step in range(25):
    rgb, depth = cam.frames()
    pose = kin.tool_pose(arm.joints())
    state = task.observe(rgb, depth, pose, arm)
    if task.problem:                      # the plan, a missed grasp, a stopped move, or the spanner's top
        print("stopping:", task.problem)
        rec.note("stopping: " + task.problem)
        break
    d = jev.choose(state, SLIDE_OVER_SKILLS, INSTRUCTIONS)
    print(step, d.skill, "confidence %.2f" % (d.confidence or 0))
    rec.note("%d %s %.2f" % (step, d.skill, d.confidence or 0))

    ok, why = gov.check_confidence(d.skill, d.confidence)
    if ok:
        ok, why = gov.check_precondition(d.skill, state)
    if not ok:
        print("   refused:", why)
        rec.note("refused: " + why)
        continue
    if d.skill == "done":
        break
    reached, why = task.act(d.skill)
    if why:
        print("   stopped:", why)
        rec.note("stopped: " + why)

print("safety events:", gov.summary())
if task.bottom is not None:
    print("lined up to %.1f mm in %d looks; the fitting's bottom went down to %.0f mm above the table "
          "(the spanner's top: %.0f mm)" % (1000 * task.error, task.looks, 1000 * task.bottom, 1000 * task.top))
print("jev calls:", jev.calls, "mean %.0f ms" % (1000*jev.seconds/max(jev.calls,1)),
      "cost $%.4f" % jev.cost)

In [ ]:
task.finish(runner, arm)   # leaves the gripper empty: a fitting still held goes back where it was picked
arm.close()
rec.close()
cam.close()